# DMSF Training — Google Colab

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**

**Chuẩn bị một lần duy nhất:**
1. Zip folder `baseline_DMSF/`:
   ```
   # Windows: chuột phải baseline_DMSF → Send to → Compressed (zipped) folder
   ```
2. Upload `baseline_DMSF.zip` lên **Google Drive** (MyDrive)
3. Không cần upload dataset thủ công — Cell 5 tự download COCO train2017/val2017 +
   annotations từ http://images.cocodataset.org/ rồi convert sang 10-class VisDrone
   scheme (chỉ giữ ảnh có person/bicycle/car/truck/bus/motorcycle), lưu kết quả vào Drive
   để tái dùng giữa các session.
4. Mở file này trong Colab rồi chạy từng cell

**Nếu session bị ngắt:** chạy lại từ Cell 1 — tự động resume từ checkpoint lưu trong Drive

**Ước tính thời gian trên T4:** ~3–5 giờ cho 600 epochs (chưa tính lần đầu cần download + convert COCO, ~30-60 phút)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1: Mount Google Drive
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
import os

drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/m.a/DMSF_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

print(f'✓ Drive mounted')
print(f'  Checkpoints → {CKPT_DIR}')

Mounted at /content/drive
✓ Drive mounted
  Checkpoints → /content/drive/MyDrive/m.a/DMSF_checkpoints


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2: Kiểm tra GPU
# ═══════════════════════════════════════════════════════════════
import torch
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
assert torch.cuda.is_available(), '❌ GPU không khả dụng — đổi Runtime type sang T4!'

Tesla T4, 15360 MiB, 580.82.07
PyTorch : 2.11.0+cu128
CUDA    : 12.8


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3: Cài thư viện
# ═══════════════════════════════════════════════════════════════
!pip install -q opencv-python-headless scipy tqdm pyyaml ultralytics
print('✓ Dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 77.5 MB/s eta 0:00:00
✓ Dependencies installed


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 4: Giải nén code DMSF
# ═══════════════════════════════════════════════════════════════
import os, sys

ZIP_PATH = '/content/drive/MyDrive/m.a/baseline_DMSF.zip'
CODE_DIR = '/content/baseline_DMSF'

assert os.path.exists(ZIP_PATH), (
    f'Không tìm thấy {ZIP_PATH}\n'
    f'→ Upload baseline_DMSF.zip vào MyDrive rồi chạy lại'
)

if not os.path.isdir(CODE_DIR):
    !unzip -q "{ZIP_PATH}" -d /content/
    print('✓ Code extracted')
else:
    print('✓ Code đã có (bỏ qua giải nén)')

os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f'  Working dir: {os.getcwd()}')
!ls -la

✓ Code extracted
  Working dir: /content/baseline_DMSF
total 2024
drwxrwxrwx 5 root root    4096 May 29 10:31 .
drwxr-xr-x 1 root root    4096 Jun  2 08:58 ..
-rw-rw-rw- 1 root root     274 May 24 18:56 check_ckpt.py
-rw-rw-rw- 1 root root    4929 May 29 10:31 CLAUDE.md
-rw-rw-rw- 1 root root    5070 May 20 18:49 cloud_inference.py
-rw-rw-rw- 1 root root     820 May 20 18:50 config.yaml
-rw-rw-rw- 1 root root 1949622 May 18 20:47 DMSF_A_Dynamic_Model_Splitting_Framework_for_Edge-Cloud_Collaborative_Inference.pdf
-rw-rw-rw- 1 root root   22270 May 29 09:51 DMSF_Colab.ipynb
-rw-rw-rw- 1 root root   11999 May 28 16:00 DMSF_Kaggle.ipynb
-rw-rw-rw- 1 root root    7373 May 20 18:49 edge_inference.py
-rw-rw-rw- 1 root root    4707 May 20 18:50 evaluate.py
drwxrwxrwx 7 root root    4096 May 23 18:12 .git
drwxrwxrwx 3 root root    4096 May 29 10:29 models
-rw-rw-rw- 1 root root    3811 May 29 10:30 README.md
-rw-rw-rw- 1 root root     107 May 20 18:50 requirements.txt
-rw-rw-rw- 1 root root    

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 5: Chuẩn bị COCO → 10-class vehicle/pedestrian (VisDrone scheme)
# Dataset (đã convert, đã lọc) lưu vào Drive để tái dùng giữa các session.
# Raw COCO (~19GB) được tự download trực tiếp, không cần upload tay.
# ═══════════════════════════════════════════════════════════════
import os

DRIVE_DATA   = '/content/drive/MyDrive/m.a/DMSF_checkpoints/data/coco_vehicle10'
LOCAL_DATA   = '/content/coco_vehicle10'
DONE_FLAG    = DRIVE_DATA + '/.done'

def _count(split, base):
    p = f'{base}/images/{split}'
    return len(os.listdir(p)) if os.path.isdir(p) else 0

# ── Nếu đã có local copy thì không làm gì cả ────────────────────────────
if _count('train', LOCAL_DATA) > 0:
    print(f'✓ COCO-vehicle10 local đã sẵn sàng: {_count("train", LOCAL_DATA)} train | {_count("val", LOCAL_DATA)} val')

# ── Nếu Drive đã có data (flag .done) thì copy xuống local SSD ──────────
elif os.path.exists(DONE_FLAG) and _count('train', DRIVE_DATA) > 0:
    print(f'✓ COCO-vehicle10 trên Drive: {_count("train", DRIVE_DATA)} train — copy xuống local SSD...')
    import shutil
    shutil.copytree(DRIVE_DATA, LOCAL_DATA)
    print(f'✓ Copy xong: {_count("train", LOCAL_DATA)} train | {_count("val", LOCAL_DATA)} val')

# ── Lần đầu: download trực tiếp + convert ───────────────────────────────
else:
    RAW = '/content/coco_raw'
    os.makedirs(RAW, exist_ok=True)

    print('Downloading COCO train2017 images (~18 GB)...')
    !wget -q -O "{RAW}/train2017.zip" http://images.cocodataset.org/zips/train2017.zip
    print('Downloading COCO val2017 images (~1 GB)...')
    !wget -q -O "{RAW}/val2017.zip" http://images.cocodataset.org/zips/val2017.zip
    print('Downloading annotations (~241 MB)...')
    !wget -q -O "{RAW}/ann.zip" http://images.cocodataset.org/annotations/annotations_trainval2017.zip

    print('Extracting...')
    !unzip -q "{RAW}/train2017.zip" -d "{RAW}/"
    !unzip -q "{RAW}/val2017.zip" -d "{RAW}/"
    !unzip -q "{RAW}/ann.zip" -d "{RAW}/"

    print('Converting COCO → 10-class YOLO format (chỉ giữ ảnh có person/vehicle)...')
    from utils.coco_vehicle10 import prepare_coco_vehicle10
    os.makedirs(DRIVE_DATA, exist_ok=True)
    prepare_coco_vehicle10(coco_root=RAW, dst_root=DRIVE_DATA)

    assert _count('train', DRIVE_DATA) > 0, '❌ Convert thất bại — kiểm tra lại download COCO'
    open(DONE_FLAG, 'w').close()

    import shutil
    shutil.copytree(DRIVE_DATA, LOCAL_DATA)
    print(f'✓ Done: {_count("train", LOCAL_DATA)} train | {_count("val", LOCAL_DATA)} val')

# ── Tạo symlink cho code nội bộ (Cell 7 evaluate dùng) ──────────────────
os.makedirs('/content/baseline_DMSF/data', exist_ok=True)
if not os.path.exists('/content/baseline_DMSF/data/coco_vehicle10'):
    os.symlink(LOCAL_DATA, '/content/baseline_DMSF/data/coco_vehicle10')

✓ VisDrone trên Drive: 6471 train — copy xuống local SSD...


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 6: Training (auto-resume từ Drive checkpoint)
# Muốn train lại từ đầu: xoá file checkpoint tương ứng trong
#   MyDrive/DMSF_checkpoints/ rồi chạy lại cell này
#
#   yolov5s           → xoá last.pt              và best.pt
#   yolo26n_dmsf       → xoá last_dmsf26n.pt       và best_dmsf26n.pt       (VisDrone, nc=10)
#   yolo26n_dmsf_coco  → xoá last_dmsf26n_coco.pt  và best_dmsf26n_coco.pt  (COCO-vehicle, nc=80)
# ═══════════════════════════════════════════════════════════════
import os, sys, shutil, time, random
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from pathlib import Path

# ── Chọn model ──────────────────────────────────────────────────────────
MODEL_TYPE = 'yolo26n_dmsf_coco'   # 'yolov5s' | 'yolo26n_dmsf' (VisDrone) | 'yolo26n_dmsf_coco' (COCO-vehicle)
# ────────────────────────────────────────────────────────────────────────

EPOCHS    = 100   # COCO-vehicle có nhiều ảnh/epoch hơn VisDrone (~10x) nên không cần 670 epoch
BATCH     = 32
IMGSZ     = 640
NC        = 80    # giữ nguyên index COCO gốc (person=0, car=2, motorcycle=3, bus=5, truck=7...)
LR        = 0.01
WORKERS   = 2
VAL_FREQ  = 10
SAVE_FREQ = 10
DEVICE    = torch.device('cuda:0')
CKPT_DIR  = '/content/drive/MyDrive/m.a/DMSF_checkpoints'
LOCAL_DIR = Path('runs/train')
DATA_DIR  = '/content/coco_vehicle10'
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

from utils.visdrone import VisDroneDataset
from utils.loss import ComputeLoss
from utils.metrics import evaluate

# ── Khởi tạo model ──────────────────────────────────────────────────────
if MODEL_TYPE == 'yolov5s':
    from models.dmsf import DMSF
    model     = DMSF(nc=NC).to(DEVICE)
    ckpt_name = ''           # → last.pt / best.pt
elif MODEL_TYPE == 'yolo26n_dmsf':
    from models.dmsf_26n import DMSF26n
    model     = DMSF26n(nc=NC).to(DEVICE)
    ckpt_name = '_dmsf26n'   # → last_dmsf26n.pt / best_dmsf26n.pt
else:  # yolo26n_dmsf_coco
    from models.dmsf_26n import DMSF26n
    model     = DMSF26n(nc=NC).to(DEVICE)
    ckpt_name = '_dmsf26n_coco'   # → last_dmsf26n_coco.pt / best_dmsf26n_coco.pt

start_epoch  = 0
best_map50   = 0.0
best_map5095 = 0.0
ckpt = None

drive_last = Path(CKPT_DIR) / f'last{ckpt_name}.pt'
local_last = LOCAL_DIR      / f'last{ckpt_name}.pt'

if drive_last.exists():
    shutil.copy(drive_last, local_last)
    ckpt = torch.load(local_last, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model'], strict=False)
    start_epoch  = ckpt.get('epoch', 0)
    best_map50   = ckpt.get('map50', 0.0)
    best_map5095 = ckpt.get('map50_95', 0.0)
    print(f'↓ Resumed [{MODEL_TYPE}] từ epoch {start_epoch}  '
          f'(best mAP@50={best_map50:.4f}  mAP@50:95={best_map5095:.4f})')
else:
    if MODEL_TYPE == 'yolov5s':
        pretrained = 'yolov5s.pt'
        if not os.path.exists(pretrained):
            print('Downloading pretrained YOLOv5s...')
            !wget -q https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5s.pt
        model.load_yolov5s_weights(pretrained)
        print('▶ Train DMSF-YOLOv5s với pretrained backbone')
    else:
        print('▶ Train DMSF-YOLO26n với pretrained backbone (tự download nếu cần)...')
        model.load_yolo26n_weights('yolo26n.pt')

train_ds = VisDroneDataset(f'{DATA_DIR}/images/train', IMGSZ, augment=True)
val_ds   = VisDroneDataset(f'{DATA_DIR}/images/val',   IMGSZ, augment=False)

# ── Bỏ comment 2 dòng dưới để test nhanh (~25s/epoch thay vì 300s) ──
#train_ds = Subset(train_ds, range(500))
#val_ds   = Subset(val_ds,   range(100))

train_loader = DataLoader(train_ds, BATCH, shuffle=True,  num_workers=WORKERS,
                          pin_memory=True, persistent_workers=True,
                          collate_fn=VisDroneDataset.collate_fn, drop_last=True)
val_loader   = DataLoader(val_ds,   BATCH//2, shuffle=False, num_workers=2,
                          pin_memory=True, persistent_workers=True,
                          collate_fn=VisDroneDataset.collate_fn)

pg0, pg1, pg2 = [], [], []
for n, p in model.named_parameters():
    if not p.requires_grad: continue
    if '.bias'   in n:                        pg2.append(p)
    elif '.weight' in n and '.bn' not in n:   pg1.append(p)
    else:                                     pg0.append(p)

optimizer = optim.SGD(pg0, lr=LR, momentum=0.937, nesterov=True)
optimizer.add_param_group({'params': pg1, 'weight_decay': 5e-4})
optimizer.add_param_group({'params': pg2, 'weight_decay': 0.0})

if ckpt is not None and 'optimizer' in ckpt:
    optimizer.load_state_dict(ckpt['optimizer'])

scheduler = optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda x: (1 - x / EPOCHS) * 0.9 + 0.1,
    last_epoch=start_epoch - 1
)
compute_loss = ComputeLoss(model, nc=NC)

print(f'\nBắt đầu train [{MODEL_TYPE}]: epoch {start_epoch+1} → {EPOCHS}')
print(f'Dataset: {len(train_ds)} train | {len(val_ds)} val  |  Batch={BATCH}')
print('─' * 65)

for epoch in range(start_epoch, EPOCHS):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    for imgs, targets, _ in train_loader:
        imgs    = imgs.to(DEVICE).float()
        targets = targets.to(DEVICE)
        sp = random.choice(model.split_points)
        optimizer.zero_grad()
        loss, _ = compute_loss(model(imgs, split_point=sp), targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    elapsed  = time.time() - t0
    print(f'Epoch {epoch+1:4d}/{EPOCHS}  loss={avg_loss:.4f}  '
          f'lr={optimizer.param_groups[0]["lr"]:.5f}  {elapsed:.0f}s', flush=True)

    if (epoch + 1) % VAL_FREQ == 0:
        m = evaluate(model, val_loader, DEVICE, split_point=10, img_size=IMGSZ)
        print(f'  ↳ [Val] mAP@50={m["map50"]:.4f}  mAP@50:95={m["map50_95"]:.4f}')
        if m['map50'] > best_map50:
            best_map50   = m['map50']
            best_map5095 = m['map50_95']
            torch.save({'epoch': epoch+1, 'model': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'map50': best_map50, 'map50_95': best_map5095},
                       LOCAL_DIR / f'best{ckpt_name}.pt')
            shutil.copy(LOCAL_DIR / f'best{ckpt_name}.pt',
                        f'{CKPT_DIR}/best{ckpt_name}.pt')
            print(f'  ↳ ✓ best{ckpt_name}.pt saved  '
                  f'(mAP@50={best_map50:.4f}  mAP@50:95={best_map5095:.4f})')
        model.train()

    if (epoch + 1) % SAVE_FREQ == 0:
        torch.save({'epoch': epoch+1, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'map50': best_map50, 'map50_95': best_map5095},
                   LOCAL_DIR / f'last{ckpt_name}.pt')
        shutil.copy(LOCAL_DIR / f'last{ckpt_name}.pt',
                    f'{CKPT_DIR}/last{ckpt_name}.pt')
        print(f'  ↳ ✓ Checkpoint synced to Drive (epoch {epoch+1})')

print(f'\n✓ Training hoàn tất [{MODEL_TYPE}]. '
      f'Best mAP@50={best_map50:.4f}  mAP@50:95={best_map5095:.4f}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 7: Evaluate tất cả split points — DMSF26n
# ═══════════════════════════════════════════════════════════════
import sys, torch
from torch.utils.data import DataLoader

from models.dmsf_26n import DMSF26n
from utils.visdrone import VisDroneDataset
from utils.metrics import evaluate
from utils.split_selector import LayerProfiler, SplitSelector, feature_bytes_1bit

DEVICE   = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
CKPT_DIR = '/content/drive/MyDrive/m.a/DMSF_checkpoints'

model = DMSF26n(nc=80).to(DEVICE)
ckpt  = torch.load(f'{CKPT_DIR}/best_dmsf26n_coco.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model'], strict=False)
model.eval()
print(f'Loaded best_dmsf26n_coco.pt — epoch={ckpt["epoch"]}, mAP@50={ckpt["map50"]:.4f}  mAP@50:95={ckpt["map50_95"]:.4f}')

val_ds = VisDroneDataset('data/coco_vehicle10/images/val', 640, augment=False)
val_loader = DataLoader(val_ds, 16, shuffle=False, num_workers=2,
                        pin_memory=True, collate_fn=VisDroneDataset.collate_fn)
print(f'Val set: {len(val_ds)} anh\n')

print(f'{"Split":>6}  {"mAP@50":>8}  {"mAP@50:95":>10}  {"Feat(KB)":>9}')
print('─' * 42)
for sp in [3, 5, 7, 10]:
    m  = evaluate(model, val_loader, DEVICE, split_point=sp, img_size=640)
    kb = feature_bytes_1bit(sp) / 1024
    print(f'{sp:>6}  {m["map50"]:>8.4f}  {m["map50_95"]:>10.4f}  {kb:>9.1f}')


Loaded best_dmsf26n.pt — epoch=670, mAP@50=0.1721  mAP@50:95=0.0912
Val set: 548 anh

 Split    mAP@50   mAP@50:95   Feat(KB)
──────────────────────────────────────────


Evaluating: 100%|██████████| 35/35 [00:18<00:00,  1.93it/s]


     3    0.1422      0.0740       26.0


Evaluating: 100%|██████████| 35/35 [00:16<00:00,  2.13it/s]


     5    0.1651      0.0888       14.5


Evaluating: 100%|██████████| 35/35 [00:16<00:00,  2.18it/s]


     7    0.1598      0.0844       10.2


Evaluating: 100%|██████████| 35/35 [00:15<00:00,  2.19it/s]


    10    0.1721      0.0912        5.1


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 9: DMSF Split Inference — đo FPS/latency trên video.mp4
# Chạy edge + cloud cùng lúc trên localhost, đo từng split point
# ═══════════════════════════════════════════════════════════════
import os, time, socket, subprocess, threading, torch, cv2
import numpy as np
from pathlib import Path

from models.dmsf_26n import DMSF26n
from utils.metrics import non_max_suppression
from utils.split_selector import feature_bytes_1bit

DEVICE   = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
CKPT_DIR = '/content/drive/MyDrive/m.a/DMSF_checkpoints'
VIDEO    = '/content/video.mp4'
IMGSZ    = 640
BATCH    = 1
NC       = 8
CONF     = 0.001
IOU      = 0.6

# Copy video xuong local neu chua co
if not os.path.exists(VIDEO):
    import shutil
    print('Copying video.mp4...')
    shutil.copy2(f'{CKPT_DIR}/../video.mp4', VIDEO)

# Load model
model = DMSF26n(nc=NC).to(DEVICE)
ckpt  = torch.load(f'{CKPT_DIR}/best_dmsf26n.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model'], strict=False)
model.eval()
print(f'Loaded best_dmsf26n.pt — epoch={ckpt["epoch"]}')

def letterbox(img, size=640):
    h, w = img.shape[:2]
    r = size / max(h, w)
    nh, nw = int(h * r), int(w * r)
    img = cv2.resize(img, (nw, nh))
    canvas = np.full((size, size, 3), 114, dtype=np.uint8)
    canvas[:nh, :nw] = img
    return canvas

def load_video_frames(path, imgsz):
    cap = cv2.VideoCapture(path)
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = letterbox(frame, imgsz)
        t = torch.from_numpy(frame[:,:,::-1].copy()).float() / 255.0
        frames.append(t.permute(2,0,1))
    cap.release()
    return frames

print('Loading video frames...')
all_frames = load_video_frames(VIDEO, IMGSZ)
print(f'  {len(all_frames)} frames')

# Measure per split point
print(f'\n{"Split":>6}  {"FPS":>8}  {"Edge(ms)":>10}  {"Cloud(ms)":>10}  {"Feat(KB)":>9}')
print('─' * 55)

for sp in [3, 5, 7, 10]:
    edge_times, cloud_times = [], []

    with torch.no_grad():
        i = 0
        while i < len(all_frames):
            batch = torch.stack(all_frames[i:i+BATCH], 0).to(DEVICE)
            i += BATCH

            # Edge
            t0 = time.perf_counter()
            payload = model.forward_edge(batch, sp)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            edge_times.append((time.perf_counter() - t0) * 1000)

            # Cloud
            t0 = time.perf_counter()
            out = model.forward_cloud(payload, device=str(DEVICE))
            pred = out[0] if isinstance(out, tuple) else out
            non_max_suppression(pred, CONF, IOU)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            cloud_times.append((time.perf_counter() - t0) * 1000)

    import numpy as np_
    avg_edge  = np_.mean(edge_times)
    avg_cloud = np_.mean(cloud_times)
    total_ms  = avg_edge + avg_cloud
    fps       = 1000 / total_ms * BATCH
    kb        = feature_bytes_1bit(sp) / 1024

    print(f'{sp:>6}  {fps:>8.2f}  {avg_edge:>10.2f}  {avg_cloud:>10.2f}  {kb:>9.1f}')

print('─' * 55)
print('(Chua tinh network transmission time)')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 8: Download model về máy
# ═══════════════════════════════════════════════════════════════
from google.colab import files
CKPT_DIR = '/content/drive/MyDrive/m.a/DMSF_checkpoints'
files.download(f'{CKPT_DIR}/best.pt')
files.download(f'{CKPT_DIR}/last.pt')